In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_target = 'pitch_type'

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

# most common pitch types
list_str_pitch_type = [
    'FF',
    'SL',
    'SI',
    'FT',
    'CH',
    'CU',
    'FC',
    'FS',
]

#### Functions

In [ ]:
class SituationFeatures:
    def __init__(self):
        pass
    def fit(self, X):
        print('Situation features...')
        return self
    def transform(self, X):
        print('Situation features...')
        X = X.copy()
        # score differential from pitching team perspective
        X['score_diff'] = np.where(
            X['top'] == 1,
            X['home_team_runs'] - X['away_team_runs'],
            X['away_team_runs'] - X['home_team_runs']
        )
        # runners on base
        X['on_1b_flag'] = X['on_1b'].notna().astype(int)
        X['on_2b_flag'] = X['on_2b'].notna().astype(int)
        X['on_3b_flag'] = X['on_3b'].notna().astype(int)
        X['runners_on'] = X['on_1b_flag'] + X['on_2b_flag'] + X['on_3b_flag']
        # handedness matchup
        X['same_hand'] = (X['p_throws'] == X['stand']).astype(int)
        # is first pitch of at-bat
        X['is_first_pitch'] = (X['pcount_at_bat'] == 1).astype(int)
        return X

In [ ]:
class LagFeatures:
    def __init__(self):
        pass
    def fit(self, X):
        print('Lag features...')
        return self
    def transform(self, X):
        print('Lag features...')
        X = X.copy()
        X = X.sort_values(['game_pk', 'at_bat_num', 'pcount_at_bat'])
        # previous pitch type in this at-bat
        X['prev_pitch_type'] = X.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(1)
        X['prev_pitch_type'] = X['prev_pitch_type'].fillna('NONE')
        # 2nd previous pitch type in this at-bat
        X['prev_pitch_type_2'] = X.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(2)
        X['prev_pitch_type_2'] = X['prev_pitch_type_2'].fillna('NONE')
        return X

In [ ]:
class PitcherStats:
    def __init__(self):
        pass
    def fit(self, X):
        print('Computing pitcher stats...')
        # overall pitch type distribution per pitcher
        df_pitcher_mix = pd.crosstab(X['pitcher_id'], X['pitch_type'], normalize='index')
        df_pitcher_mix.columns = [f'pitcher_pct_{c}' for c in df_pitcher_mix.columns]
        self.df_pitcher_mix = df_pitcher_mix
        # pitch type distribution per pitcher per count
        X_tmp = X.copy()
        X_tmp['count_str'] = X_tmp['balls'].astype(str) + '-' + X_tmp['strikes'].astype(str)
        df_pitcher_count = pd.crosstab(
            [X_tmp['pitcher_id'], X_tmp['count_str']],
            X_tmp['pitch_type'],
            normalize='index'
        )
        df_pitcher_count.columns = [f'pitcher_count_pct_{c}' for c in df_pitcher_count.columns]
        self.df_pitcher_count = df_pitcher_count
        # total pitches thrown by pitcher
        self.sr_pitcher_volume = X.groupby('pitcher_id').size().rename('pitcher_total_pitches')
        # pitcher repertoire size
        self.sr_pitcher_n_types = X.groupby('pitcher_id')['pitch_type'].nunique().rename('pitcher_n_types')
        return self
    def transform(self, X):
        print('Applying pitcher stats...')
        X = X.copy()
        X['count_str'] = X['balls'].astype(str) + '-' + X['strikes'].astype(str)
        # overall pitcher mix
        X = X.merge(self.df_pitcher_mix, left_on='pitcher_id', right_index=True, how='left')
        # pitcher mix per count
        X = X.merge(self.df_pitcher_count, left_on=['pitcher_id', 'count_str'], right_index=True, how='left')
        # pitcher volume and repertoire
        X = X.merge(self.sr_pitcher_volume, left_on='pitcher_id', right_index=True, how='left')
        X = X.merge(self.sr_pitcher_n_types, left_on='pitcher_id', right_index=True, how='left')
        return X

In [ ]:
class PrepareFeatures:
    def __init__(self, list_numeric, list_categorical):
        self.list_numeric = list_numeric
        self.list_categorical = list_categorical
    def fit(self, X):
        print('Preparing features...')
        # learn pitcher mix and count mix column names from training data
        self.list_pitcher_mix = [c for c in X.columns if c.startswith('pitcher_pct_')]
        self.list_pitcher_count_mix = [c for c in X.columns if c.startswith('pitcher_count_pct_')]
        # learn aligned column list from training data
        X_num = X[self.list_numeric].copy()
        list_mix_cols = self.list_pitcher_mix + self.list_pitcher_count_mix
        list_mix_present = [c for c in list_mix_cols if c in X.columns]
        X_mix = X[list_mix_present].fillna(0).copy()
        X_cat = pd.get_dummies(X[self.list_categorical], columns=self.list_categorical, dtype=int)
        X_combined = pd.concat([X_num, X_mix, X_cat], axis=1).fillna(0)
        self.list_all_cols = sorted(X_combined.columns.tolist())
        return self
    def transform(self, X):
        print('Preparing features...')
        X = X.copy()
        # target
        y = X['pitch_type'].copy()
        # numeric
        X_num = X[self.list_numeric].copy()
        # pitcher mix (fill NaN with 0 for unseen pitchers)
        list_mix_cols = self.list_pitcher_mix + self.list_pitcher_count_mix
        list_mix_present = [c for c in list_mix_cols if c in X.columns]
        X_mix = X[list_mix_present].fillna(0).copy()
        # categorical -> one-hot
        X_cat = pd.get_dummies(X[self.list_categorical], columns=self.list_categorical, dtype=int)
        # combine
        X_out = pd.concat([X_num, X_mix, X_cat], axis=1).fillna(0)
        # align columns to training set
        X_out = X_out.reindex(columns=self.list_all_cols, fill_value=0)
        return X_out

In [ ]:
class PreprocessingModel:
    def __init__(self, list_cls_transformers):
        self.list_cls_transformers = list_cls_transformers
    def transform(self, X):
        for cls_transformer in self.list_cls_transformers:
            X = cls_transformer.transform(X)
        return X

#### Import Data from S3

In [ ]:
%%time

df_train = pd.read_csv(f's3://{str_bucket}/02_data_split/train.csv')
df_valid = pd.read_csv(f's3://{str_bucket}/02_data_split/valid.csv')
df_test = pd.read_csv(f's3://{str_bucket}/02_data_split/test.csv')

print(f'Train: {df_train.shape}')
print(f'Valid: {df_valid.shape}')
print(f'Test:  {df_test.shape}')

#### Fit and Transform on Training Data

In [ ]:
# situation features
cls_situation = SituationFeatures()
cls_situation.fit(df_train)
df_train = cls_situation.transform(df_train)

In [ ]:
# lag features
cls_lag = LagFeatures()
cls_lag.fit(df_train)
df_train = cls_lag.transform(df_train)

In [ ]:
# pitcher stats
cls_pitcher = PitcherStats()
cls_pitcher.fit(df_train)
df_train = cls_pitcher.transform(df_train)

In [ ]:
# prepare features
list_numeric = [
    'inning', 'top', 'outs', 'balls', 'strikes', 'fouls',
    'pcount_at_bat', 'pcount_pitcher',
    'score_diff',
    'on_1b_flag', 'on_2b_flag', 'on_3b_flag', 'runners_on',
    'same_hand', 'is_first_pitch',
    'pitcher_total_pitches', 'pitcher_n_types',
]
list_categorical = ['p_throws', 'stand', 'prev_pitch_type', 'prev_pitch_type_2']

cls_prepare = PrepareFeatures(list_numeric=list_numeric, list_categorical=list_categorical)
cls_prepare.fit(df_train)
X_train = cls_prepare.transform(df_train)
y_train = df_train[str_target].copy()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')

#### Build PreprocessingModel

In [ ]:
list_cls_transformers = [
    cls_situation,
    cls_lag,
    cls_pitcher,
    cls_prepare,
]
cls_model_preprocessing = PreprocessingModel(
    list_cls_transformers=list_cls_transformers,
)

#### Transform Valid and Test

In [ ]:
X_valid = cls_model_preprocessing.transform(df_valid)
y_valid = df_valid[str_target].copy()

X_test = cls_model_preprocessing.transform(df_test)
y_test = df_test[str_target].copy()

print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

In [ ]:
# verify feature columns
print(f'Aligned feature count: {len(X_train.columns)}')
print('\nFeature columns:')
for col in sorted(X_train.columns):
    print(f'  {col}')

#### Save to Local Output

In [ ]:
# save preprocessing model locally
str_filename = 'cls_model_preprocessing.joblib'
str_local_path = f'{str_dirname_output}/{str_filename}'
joblib.dump(cls_model_preprocessing, str_local_path)
print(f'Saved preprocessing model to {str_local_path}')

# save feature matrices and targets to S3
for str_name, df_x, sr_y in [('train', X_train, y_train), ('valid', X_valid, y_valid), ('test', X_test, y_test)]:
    df_x.to_csv(f's3://{str_bucket}/{str_task}/X_{str_name}.csv', index=False)
    sr_y.to_csv(f's3://{str_bucket}/{str_task}/y_{str_name}.csv', index=False)
    print(f'Saved X_{str_name}.csv ({df_x.shape}) and y_{str_name}.csv ({sr_y.shape}) to S3')